<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/5_ALM_AnomalyDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Time-Series Modeling & Foundation Models

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell prepares everything; the backend is started below.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand why time-series analysis matters** for asset management
2. **Compare traditional vs foundation model** approaches to anomaly detection
3. **Explain foundation models** for time-series (MOMENT, NV-Tesseract)
4. **Use the anomaly detection tool** integrated in NAT to detect sensor anomalies
5. **Analyze multi-engine anomaly patterns** across engines with different health profiles
6. **Correlate anomalies with Remaining Useful Life** to understand degradation signatures
7. **Understand when to use** foundation models vs task-specific models

## Anomaly Detection in This Lab

The `anomaly_detection` tool in `config-reasoning.yml` uses **NV-Tesseract**, NVIDIA's production-grade time-series foundation model, served via NIM. An open-source alternative, **MOMENT**, can also be configured by swapping the tool type in the config.

> **🟢 Colab edition — anomaly detection model.** The NV-Tesseract NIM described below is a self-hosted container that requires NGC approval and a GPU, so it is not available as a hosted endpoint. In this edition the `anomaly_detection` tool is configured with the open-source **MOMENT-1-small** foundation model (`_type: moment_anomaly_detection_tool`), which runs locally on CPU or GPU. NV-Tesseract builds on the same MOMENT architecture, so everything below about how the model works still applies. The tool adds an `is_anomaly` column to the JSON file returned by the SQL retriever, and the plot tool writes `anomaly_plot_<sensor>_engine<N>.html/png`.

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

---
# Section 5.1: Time-Series Analysis in Asset Management

## Why Time-Series Matters

Asset lifecycle management is fundamentally a **time-series problem**:

### Turbofan Engine Data:
```
Cycle 1:  [sensor1: 518.67, sensor2: 641.82, ...] → Healthy
Cycle 50: [sensor1: 518.45, sensor2: 642.15, ...] → Still OK
Cycle 150:[sensor1: 516.23, sensor2: 645.78, ...] → Degrading
Cycle 200:[sensor1: 512.45, sensor2: 651.32, ...] → Critical!
```

The **temporal patterns** reveal degradation that single snapshots cannot.

## Key Time-Series Tasks

### 1. Anomaly Detection
Identify unusual patterns that deviate from normal operation:
- Sudden spikes or drops
- Unexpected oscillations
- Drift from baseline

### 2. Forecasting
Predict future values:
- When will this sensor exceed threshold?
- What will RUL be in 20 cycles?
- How much longer can we operate?

### 3. Pattern Recognition
Identify similar degradation patterns:
- Has this failure mode occurred before?
- Which engines show similar behavior?
- What were the outcomes?

### 4. Classification
Categorize operating states:
- Normal vs degraded
- Fault type identification
- Severity assessment

## Traditional Approaches vs Foundation Models

### Traditional Statistical Methods:

**Z-Score Anomaly Detection:**
```python
# Simple statistical approach
mean = data.mean()
std = data.std()
z_scores = (data - mean) / std
anomalies = abs(z_scores) > 3  # 3-sigma rule
```

**Limitations:**
- Assumes normal distribution
- Ignores temporal dependencies
- Fixed thresholds
- Cannot learn complex patterns

**ARIMA, Prophet, Traditional ML:**
- Good for simple patterns
- Require manual feature engineering
- Need per-dataset tuning
- Limited generalization

### Foundation Model Approach:

**Pre-trained on massive time-series data:**
```
Training Data:
- Millions of time-series from diverse domains
- Sensors, stock prices, weather, medical signals
- Learns universal temporal patterns

Then:
- Fine-tune for turbofan engines (optional)
- Or use zero-shot for anomaly detection
- Adapts to new data automatically
```

**Advantages:**
- Learns complex temporal dependencies
- Generalizes across domains
- Minimal tuning required
- Captures subtle patterns

## Real-World Impact

Foundation models for time-series enable:

- **Early Warning**: Detect degradation weeks before failure
- **Root Cause**: Identify which sensors indicate specific faults
- **Transfer Learning**: Apply learning from one asset type to another
- **Few-Shot Learning**: Work with limited failure examples

---
# Section 5.2: Foundation Models for Time-Series

## Overview of Time-Series Foundation Models

Similar to how GPT-4 is a foundation model for text and CLIP for images, new foundation models tackle time-series:

### Key Models in 2024-2025:

1. **MOMENT-1-Large** (Carnegie Mellon + MIT)
   - 385M parameters
   - Trained on Time-Series Pile (diverse domains)
   - Tasks: Forecasting, classification, anomaly detection
   - Open source

2. **TimesFM** (Google)
   - 200M parameters
   - Focus on forecasting
   - Pre-trained on 100B real-world time points

3. **NVIDIA NV-Tesseract**
   - Multi-modal time-series understanding
   - Industrial sensor focus
   - Part of NVIDIA AI Enterprise

## NVIDIA NV-Tesseract

### What is NV-Tesseract?

**NV-Tesseract** is NVIDIA's time-series foundation model designed for industrial applications. It builds on the **MOMENT** architecture and uses a **T5 encoder** backbone to solve a **masked-reconstruction** pre-training task — hide portions of a time series and ask the model to reconstruct them — which gives it a deep understanding of temporal patterns.

### Architecture (MOMENT backbone + T5 Encoder):

```
Raw sensor time series
        │
        ▼
Reversible Instance Normalization
  (normalize using mean/variance of the sequence;
   denormalize output with the same stats)
        │
        ▼
Patch Segmentation
  (split into fixed-length patches)
        │
        ▼
Masking + Linear Projection → Embeddings
  ├─ Observed patches  → Patch embeddings
  └─ Masked patches    → Learnable mask embeddings
        │
        ▼
T5 Transformer Encoder
  • Layer norm without additive bias (fewer params, less overfitting)
  • Normalization before residual skip connections (stable gradients)
  • Relative positional embeddings (learnable, distance-based)
  • Absolute positional embeddings added per patch (sinusoidal)
        │
        ▼
Task Head (prediction / classification / anomaly)
  → Reconstructs masked & unmasked patches
```

**Training objective:** Masked-reconstruction with MSE + MAE loss — the model learns to fill in missing segments, which teaches it what "normal" temporal patterns look like for a given signal.

### Why This Architecture Works for Anomaly Detection:

- **Reversible instance normalization** ensures the model handles varying sensor scales without domain-specific preprocessing
- **Patch-based processing** captures local temporal structure (e.g., a sensor's behavior over 10–50 cycles)
- **The T5 encoder's relative positional embeddings** let the model reason about "how far apart" events are, not just "which timestep" — critical for detecting degradation that unfolds gradually
- **Masked reconstruction** training means the model learns what "should" come next; when reality deviates significantly, that's an anomaly

### Key Capabilities:

| Capability | Details |
|---|---|
| **Multi-task** | Anomaly detection, forecasting, classification from one pre-trained model |
| **Multivariate** | Handles multiple sensor channels simultaneously |
| **Zero-shot** | Use pre-trained weights directly on new data without fine-tuning |
| **Few-shot** | Adapt with as few as 10–100 labeled examples |
| **NIM deployment** | Served as a production-grade NVIDIA NIM microservice |

### Comparison with Task-Specific Models:

| Aspect | NV-Tesseract (Foundation) | XGBoost (Task-Specific) |
|--------|---------------------------|-------------------------|
| **Training Data** | Millions of diverse time-series | Only turbofan data |
| **Generalization** | Works across domains | Turbofan-specific |
| **Temporal Patterns** | Learns complex dependencies | Manual features |
| **Adaptation** | Fast fine-tuning | Full retraining |
| **Cold Start** | Good with little data | Needs substantial data |
| **Interpretability** | Moderate (attention maps) | High (feature importance) |

### When to Use NV-Tesseract:
- ✅ Limited labeled data
- ✅ Need to generalize across assets
- ✅ Complex temporal patterns
- ✅ Novel failure modes
- ✅ Multi-sensor fusion

### When to Use Traditional ML:
- ✅ Abundant labeled data
- ✅ Well-defined features
- ✅ Interpretability required
- ✅ Low latency critical
- ✅ Resource constrained

## In This Lab

This lab uses **NV-Tesseract** for anomaly detection (configured as `nv_tesseract_anomaly_detection_tool` in `config-reasoning.yml`). NV-Tesseract is NVIDIA's production-grade time-series foundation model served via NIM, offering high accuracy and multi-task support (forecasting, classification, anomaly detection).

As an open-source alternative, the **MOMENT** foundation model can be swapped in by changing the tool type in the config. MOMENT runs locally and provides zero-shot anomaly detection without requiring a separate NIM endpoint.

---
# Section 5.3: Anomaly Detection with NV-Tesseract

## NV-Tesseract Foundation Model via NIM

The `anomaly_detection` tool uses **NV-Tesseract**, NVIDIA's time-series foundation model, served via NIM for production-grade anomaly detection.

### Model Details:
- **Architecture**: MOMENT backbone with T5 Transformer encoder, trained via masked reconstruction (MSE + MAE)
- **Processing**: Reversible instance normalization → patch segmentation → transformer encoding → task head
- **Serving**: NVIDIA NIM microservice (local endpoint at `http://localhost:8001` in this lab)
- **Tasks**: Anomaly detection, forecasting, classification (multi-task from one pre-trained model)

### How It Works:

```
Input: Sensor time-series [value1, value2, ..., valueN]
    ↓
NV-Tesseract (via NIM):
- Analyzes multivariate temporal patterns
- Applies learned industrial sensor representations
- Computes anomaly scores with auto-thresholding
    ↓
Output: Enhanced data with 'is_anomaly' boolean column
        (True = Anomaly detected)
```

### Agent Workflow:

The anomaly detection pipeline requires two steps:

1. **SQL Retrieval**: Fetch sensor data (must include `unit_number`, `time_in_cycles`, and the target sensor column)
2. **Anomaly Detection**: Pass the JSON file to the `anomaly_detection` tool
3. **Visualization** (optional): Use `plot_anomaly` to create interactive plots

### Config in `config-reasoning.yml` (Colab edition uses MOMENT):
```yaml
functions:
  # DLI lab (self-hosted NIM):
  # anomaly_detection:
  #   _type: nv_tesseract_anomaly_detection_tool
  #   nim_endpoint: "http://localhost:8001"
  #   timeout: 120
  #   output_folder: "agent_workspace"
  # NATcolab (this notebook):
  anomaly_detection:
    _type: moment_anomaly_detection_tool
    output_folder: "agent_workspace"
```

<details>
<summary><b>Alternative: MOMENT Foundation Model (open-source, local)</b></summary>

MOMENT-1-Large (385M params, Carnegie Mellon + MIT) can be used as an open-source alternative. It runs locally without requiring a NIM endpoint:
```yaml
functions:
  anomaly_detection:
    _type: moment_anomaly_detection_tool
    output_folder: "agent_workspace"
```
MOMENT uses a reconstruction-based approach — high reconstruction error indicates anomalies. It is trained on the Time-Series Pile (diverse domains: sensors, weather, medical, financial) and provides zero-shot anomaly detection without fine-tuning.

> **Note:** To use MOMENT, you must first uncomment its import in `register.py`:
> ```
> # In: asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/register.py
> from .predictors import moment_anomaly_detection_tool   # uncomment this line
> ```
> Then reinstall the package (`uv pip install -e .`) and restart the NAT backend.

</details>

Now let's use the anomaly detection tool through the NAT agent.

**Run this query with the client code below:**

In [ ]:
client.restart()

In [ ]:
client.check_health()

In [ ]:
client.query(timeout=900, message="Detect anomalies in sensor_measurement_4 for unit 78 in FD001 training data and visualize the results")

In [ ]:
# Visualize the plot
client.show("anomaly_plot_sensor_measurement_4_engine78.png")

### What to Observe:

1. The agent first retrieves sensor data via SQL (saved as JSON)
2. Then passes the JSON file to the `anomaly_detection` tool (the foundation model via NIM)
3. the foundation model analyzes temporal patterns and flags anomalies
4. The `plot_anomaly` tool creates an interactive visualization with red markers on anomalous points

### What Makes This Different from Simple Thresholds:

- the foundation model considers **temporal context** (a value isn't anomalous in isolation -- it depends on surrounding patterns)
- Applies **auto-thresholding** that adapts dynamically based on the signal's behavior
- Detects **subtle drift patterns** that fixed thresholds would miss
- Leverages **industrial sensor knowledge** learned from diverse training data

---
# Exercise 5.1: Multi-Engine Anomaly Comparison

## Objective

Compare anomaly detection results across three engines with **different health profiles** to understand how the NV-Tesseract foundation model captures degradation signatures.

## Step 1: Run Anomaly Detection on Three Engines

Run these three queries with the client (cell below). Each targets `sensor_measurement_7`, which shows clear degradation trends in FD001.

**Engine 1 -- Unit 10 (Long lifecycle, 222 cycles):**
```
Detect anomalies in sensor_measurement_7 for unit 10 in FD001 training data and visualize the results
```

**Engine 2 -- Unit 50 (Medium lifecycle, 200 cycles):**
```
Detect anomalies in sensor_measurement_7 for unit 50 in FD001 training data and visualize the results
```

**Engine 3 -- Unit 90 (Short lifecycle, 154 cycles):**
```
Detect anomalies in sensor_measurement_7 for unit 90 in FD001 training data and visualize the results
```

## Step 2: Compare Results

For each engine, examine the anomaly plot and note:

1. **Anomaly count**: How many anomalies were detected?
2. **Timing**: Do anomalies cluster at the beginning, middle, or end of the lifecycle?
3. **Pattern**: Are anomalies scattered or do they appear in bursts?
4. **Correlation with degradation**: Do more anomalies appear as the engine approaches failure?

## Step 3: Correlate with Ground Truth RUL

Run the code cell below to see the lifecycle length and RUL distribution for each engine:

In [ ]:
for unit in (10, 50, 90):
    client.query(f"Detect anomalies in sensor_measurement_7 for unit {unit} in FD001 training data and visualize the results",
                 timeout=900)
    client.show(f"anomaly_plot_sensor_measurement_7_engine{unit}.png")

In [ ]:
import sqlite3
import pandas as pd
import json, glob, os

db_path = f"{ALM_HOME}/database/nasa_turbo.db"
conn = sqlite3.connect(db_path)

def latest_anomaly_file(unit):
    """Most recent JSON in agent_workspace with an `is_anomaly` column for this engine unit.
    (The MOMENT tool adds `is_anomaly` to the JSON file produced by the SQL retriever.)"""
    for path in sorted(glob.glob(f"{ALM_HOME}/agent_workspace/*.json"), key=os.path.getmtime, reverse=True):
        try:
            df = pd.read_json(path)
        except Exception:
            continue
        if "is_anomaly" in df.columns and "unit_number" in df.columns and set(df["unit_number"].unique()) == {unit}:
            return df
    return None

units = [10, 50, 90]
print("=" * 60)
print(f"{'Unit':>6} | {'Cycles':>7} | {'Max RUL':>8} | {'Anomalies':>9} | {'Anomaly %':>9}")
print("-" * 60)

for unit in units:
    df = pd.read_sql_query(
        f"SELECT time_in_cycles FROM train_FD001 WHERE unit_number={unit}", conn)
    n_cycles = len(df)

    rul_df = pd.read_sql_query(
        f"SELECT RUL FROM training_data WHERE unit_number={unit} AND dataset='FD001'", conn)
    max_rul = rul_df['RUL'].max() if not rul_df.empty else "N/A"

    anom = latest_anomaly_file(unit)
    if anom is not None:
        n_anom = int(anom['is_anomaly'].sum())
        pct = f"{100 * n_anom / len(anom):.1f}%"
    else:
        n_anom, pct = "?", "run query first"

    print(f"{unit:>6} | {n_cycles:>7} | {max_rul:>8} | {n_anom:>9} | {pct:>9}")

conn.close()
print("=" * 60)

## Step 4: Reflect on Results

Based on the comparison table and the anomaly plots, consider:

1. **Early detection**: Does the foundation model detect degradation before simple thresholds would? Look at where the first anomalies appear relative to the engine's total lifecycle.
2. **False positives**: Are there anomalies in the early (healthy) portion of the lifecycle? If so, how many?
3. **Anomaly density**: Do engines closer to failure show more concentrated anomalies?
4. **Actionable threshold**: At what anomaly rate should a maintenance alert be triggered?

These questions illustrate why foundation models are valuable -- they capture **temporal context** that raw threshold-based methods miss.

---
# Key Takeaways

In this notebook, you learned:

1. ✅ Why time-series analysis is fundamental to asset management
2. ✅ How foundation models differ from traditional approaches
3. ✅ What NVIDIA NV-Tesseract offers for industrial time-series
4. ✅ How NV-Tesseract provides production-grade anomaly detection via NIM
5. ✅ Practical comparison of foundation vs statistical methods
6. ✅ Multi-engine anomaly analysis patterns
7. ✅ When to use foundation models vs traditional ML

## Time-Series Foundation Model Benefits

- **Generalization**: Learn once, apply to many asset types
- **Few-shot**: Work with limited failure examples
- **Transfer Learning**: Knowledge from other domains helps
- **Complex Patterns**: Capture subtle degradation signatures
- **Adaptability**: Fine-tune quickly for new scenarios


## What's Next?

The **Optional Notebook** covers **Code Generation** — an advanced capability that lets the agent write and execute custom Python code for complex queries that pre-built tools cannot handle (e.g., moving averages, custom statistical transforms, multi-axis charts).

If you want to explore code generation, continue to the Optional Notebook. Otherwise, proceed directly to the Assessment.

**[Continue to Optional Notebook - Code Generation →](Optional_ALM_Codegen.ipynb)**

**[Skip to Final Assessment →](ALM_assessment.ipynb)**